# Phase 1 — Data Understanding and Cleaning

**IB9BW0 Analytics in Practice · Universal Plus e-mail marketing**

This notebook is the interactive twin of `phase1_data_understanding.py`: the cleaning logic is identical,
only the single `main()` call is split into one cell per audit step so that each table can be read next to
the decision it justifies.

- **Input**: `campaign_data.csv` (52,000 × 21, carries `visit`), `next_campaign.csv` (12,000 × 20, no `visit`)
- **Output**: audit printed to stdout only — nothing is written to disk; two cleaned DataFrames are returned
- **Run**: `Kernel → Restart & Run All`

### How to point the notebook at the data

Set the folder that holds the two CSVs, either in the shell before launching Jupyter

```bash
export AIP_DATA_DIR="/your/folder/dataset"
```

or in the first code cell below, right after the `DATA_DIR` line

```python
import os; os.environ["AIP_DATA_DIR"] = "/your/folder/dataset"
```

`check_data_dir()` stops with an actionable message if the files are not there, instead of a bare
`FileNotFoundError` twenty lines later.

## 0 · Imports

In [1]:
import os

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

## 1 · Configuration

Every threshold and column group lives here, so nothing downstream hard-codes a number.

`BREAKEVEN_PP` is the one figure the whole assignment turns on: an e-mail costs £3.50, a store visit is
worth 93.00 × 0.30 = £28 of margin, so a campaign only pays if it lifts the visit rate by
3.50 / 28 = 12.5 percentage points.

In [2]:
# Folder that holds campaign_data.csv and next_campaign.csv.
# Override without touching the code:  export AIP_DATA_DIR="/your/folder"
DATA_DIR = os.environ.get("AIP_DATA_DIR", "path/to/dataset")

TRAIN_FILE = "campaign_data.csv"
DEPLOY_FILE = "next_campaign.csv"

COST_PER_EMAIL = 3.50                                   # cost per email sent
SPEND_PER_VISIT = 93.0                                  # average customer spend per visit
MARGIN = 0.30                                           # gross profit margin
VALUE_PER_VISIT = SPEND_PER_VISIT * MARGIN              # value per store visit = 28
BREAKEVEN_PP = COST_PER_EMAIL / VALUE_PER_VISIT * 100   # break-even increment = 12.5pp

NUM_COLS = ["recency", "purchase", "age", "satisfaction_score", "spend"]
BIN_COLS = ["mens", "womens", "new_customer", "dependent", "account",
            "employed", "phone", "payment_card"]
CAT_COLS = ["zip_area", "channel", "delivery", "marriage"]

# purchase_segment bands, used to rebuild the ordinal from the purchase amount
SEG_BINS = [0, 100, 200, 350, 500, 750, 1000, np.inf]

## 2 · Loading and inspection helpers

`load()` declares blanks as the only form of missing (`keep_default_na=False`, `na_values=[""]`), so a
literal `"None"` or `"NA"` in a cell would not be silently swallowed.

`overview()` is built on the text-level frame on purpose: `df.info()` would report all 21 columns as `str`
and reveal nothing about how values are actually encoded (`'3) 200 - 350'`, `'1,000 +'`).

In [3]:
def check_data_dir():
    """Fail early with an actionable message instead of a bare FileNotFoundError."""
    missing = [f for f in (TRAIN_FILE, DEPLOY_FILE)
               if not os.path.isfile(os.path.join(DATA_DIR, f))]
    if missing:
        raise FileNotFoundError(
            f"Could not find {', '.join(missing)} in DATA_DIR = {DATA_DIR!r}.\n"
            f"Set the environment variable AIP_DATA_DIR to the folder that holds "
            f"campaign_data.csv and next_campaign.csv, e.g.\n"
            f"    export AIP_DATA_DIR=\"path/to/dataset\""
        )


def load(name):
    """Read a CSV with an explicit definition of missing.

    keep_default_na=False stops pandas from turning 'None'/'NA' into NaN;
    na_values=[""] declares that only blank cells count as missing.
    The dataset contains no pseudo-missing codes, but this makes the check
    explicit rather than assumed.
    """
    return pd.read_csv(os.path.join(DATA_DIR, name),
                       keep_default_na=False, na_values=[""], dtype=str)

In [4]:
def overview(df, name):
    """Raw-text overview: shape, sample rows, per-column cardinality and missingness.

    Built on the text-level frame, because df.info() would report all 21
    columns as str and say nothing about how values are actually encoded.
    """
    print(f"\n=== {name} ===")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(df.head(3).to_string())

    rows = []
    for c in df.columns:
        s = df[c]
        m = s.isna()
        top = s[~m].value_counts().head(3)
        rows.append({
            "column": c,
            "dtype": str(s.dtype),
            "n_unique": int(s.nunique(dropna=True)),   # 1 = zero variance, n = identifier
            "n_missing": int(m.sum()),
            "pct_missing": round(m.mean() * 100, 1),
            "top_3_values": " | ".join(f"{k}({v:,})" for k, v in top.items()),
        })
    info = pd.DataFrame(rows).set_index("column")
    print("\n--- Column overview ---")
    print(info.to_string(max_colwidth=40))
    return info

In [5]:
def numeric_summary(df, cols):
    """Numeric summary with p01/p99, 3*IQR fences and skew.

    Quantiles rather than min/max characterise the tails; skew decides whether
    a log transform or a tree-based model is needed later.
    """
    num = df[cols].apply(pd.to_numeric, errors="coerce")
    q1, q3 = num.quantile(0.25), num.quantile(0.75)
    iqr = q3 - q1
    out = pd.DataFrame({
        "n": num.notna().sum(),
        "mean": num.mean().round(2),
        "sd": num.std().round(2),
        "min": num.min(),
        "p01": num.quantile(0.01).round(2),
        "median": num.median(),
        "p99": num.quantile(0.99).round(2),
        "max": num.max(),
        "skew": num.skew().round(2),
        "n_far_3iqr": ((num < q1 - 3 * iqr) | (num > q3 + 3 * iqr)).sum(),
    })
    print("\n--- Numeric summary ---")
    print(out.to_string())
    return out

## 3 · Quality diagnostics

`missing_profile()` must run **before** `clean()`, because it needs `spend`, which `clean()` drops as a
post-outcome variable. Each row carries its own observed-group reference, so the table is self-contained
and no baseline is hard-coded.

In [6]:
def parse_segment(df):
    """purchase_segment -> ordinal 1..7; falls back to the band implied by the amount.

    Labels in the file look like '3) 200 - 350', so the leading index is parsed
    rather than matching the range text ('7) 1,000 +' also carries a thousands
    separator and a space).
    """
    ord_seg = df["purchase_segment"].str.extract(r"^\s*(\d)\)", expand=False).astype("float")
    rebuilt = pd.cut(
        pd.to_numeric(df["purchase"], errors="coerce"),
        bins=SEG_BINS, labels=False, right=False
    ) + 1
    return ord_seg.fillna(rebuilt).astype("Int64")

In [7]:
def segment_consistency(df):
    """Cross-check the segment label against the band implied by the amount."""
    lbl = df["purchase_segment"].str.extract(r"^\s*(\d)\)", expand=False).astype("float")
    reb = pd.cut(
        pd.to_numeric(df["purchase"], errors="coerce"),
        bins=SEG_BINS, labels=False, right=False
    ) + 1
    both = lbl.notna() & reb.notna()
    return {
        "checked": int(both.sum()),
        "mismatch": int((lbl[both] != reb[both]).sum()),
        "label_missing": int(lbl.isna().sum()),
    }

In [8]:
def smd(x, flag):
    """Standardised mean difference: (mean | missing - mean | observed) / overall SD.

    |SMD| < 0.1 is conventionally read as missing-at-random; above 0.1 the
    missingness is related to customer attributes rather than to chance.
    """
    x = pd.to_numeric(x, errors="coerce")
    sd = x.std()
    if not sd or np.isnan(sd):
        return np.nan
    return (x[flag].mean() - x[~flag].mean()) / sd

In [9]:
def missing_profile(df):
    """Missingness profile: the evidence behind impute / keep-NaN decisions.

    Each row carries its own observed-group reference, so the table is
    self-contained and no reference value is hard-coded.
    Must run before clean(), which drops spend.
    """
    y = df["visit"].astype(int)
    seg = parse_segment(df).astype(float)
    purchase = pd.to_numeric(df["purchase"], errors="coerce")
    spend = pd.to_numeric(df["spend"], errors="coerce")
    new = df["new_customer"].astype(float)
    rows = []
    for c in ["satisfaction_score", "recency", "age", "zip_area", "channel"]:
        m = df[c].isna()
        rows.append({
            "column": c,
            "n_missing": int(m.sum()),
            "pct": round(m.mean() * 100, 1),
            "visit_missing": round(y[m].mean() * 100, 1),
            "visit_observed": round(y[~m].mean() * 100, 1),
            "gap_pp": round((y[m].mean() - y[~m].mean()) * 100, 1),
            "flag_auc": round(roc_auc_score(y, m.astype(int)), 3),
            "new_share": round(new[m].mean() * 100, 1),
            "new_share_obs": round(new[~m].mean() * 100, 1),
            "high_band": round(seg[m].ge(5).mean() * 100, 1),
            "high_band_obs": round(seg[~m].ge(5).mean() * 100, 1),
            "smd_purchase": round(smd(purchase, m), 2),
            "spend_if_visit": round(spend[(y == 1) & m].mean(), 1),
            "spend_if_visit_obs": round(spend[(y == 1) & ~m].mean(), 1),
        })
    prof = pd.DataFrame(rows).set_index("column")
    print("\n--- Missingness profile (missing vs observed) ---")
    print(prof.to_string())
    return prof

## 4 · Cleaning

No value is invented for a missing cell. The only "filling" is an explicit `Unknown` level for
`zip_area` and `channel`, where the blank itself carries a 26%-vs-16% visit signal.

In [10]:
def clean(df, is_training):
    """Apply the cleaning decisions.

    No value is invented for a missing cell, except an explicit 'Unknown'
    level for the two nominal variables where the blank is a real signal.
    """
    log = []
    df = df.copy()

    # 1. Identifier: not a quantity, moved out of the feature set (re-attach on submission)
    ids = df.pop("Customer_ID")

    # 2. Zero variance: account is 1 for every customer
    df["account"] = df["account"].astype("Int64")
    if df["account"].nunique(dropna=False) == 1:
        df = df.drop(columns=["account"])
        log.append(("delete", "account", "zero variance: every customer has an account"))

    # 3. Target leakage: spend is recorded after the campaign; spend>0 implies visit
    if "spend" in df.columns:
        df = df.drop(columns=["spend"])
        log.append(("delete", "spend", "post-outcome variable: spend>0 implies visit"))

    # 4. Redundancy: purchase_segment is a coarse copy of purchase, and its label is
    #    missing for some rows while the amount is always present.
    #    Keep the continuous purchase; rebuild an ordinal for reporting only
    df["purchase"] = pd.to_numeric(df["purchase"], errors="coerce")
    df["purchase_segment_ord"] = parse_segment(df)
    df = df.drop(columns=["purchase_segment"])
    log.append(("delete", "purchase_segment",
                "coarse copy of purchase; ordinal rebuilt for reporting only"))

    # 5. recency: missing group is 25.7% high band (observed 11.7%), visit 30.6% vs 15.3%
    #    Zero-filling would read as 'bought this month' (the opposite direction);
    #    median-filling would read as an average customer. NaN kept.
    df["recency"] = pd.to_numeric(df["recency"], errors="coerce")
    df["recency_missing"] = df["recency"].isna().astype(int)
    log.append(("keep-nan", "recency", "missing concentrated in high-value customers; imputation forbidden"))

    # 6. age: 86.1% of missing rows are new customers (overall 48.8%) - incomplete profiles
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age_missing"] = df["age"].isna().astype(int)
    log.append(("keep-nan", "age", "structural missingness: new customers with incomplete profiles"))

    # 7. satisfaction_score: 59.8% missing = survey non-response; imputation invents an attitude
    #    No -1 sentinel (it would break the 2..10 ordinal scale). NaN kept.
    df["satisfaction_score"] = pd.to_numeric(df["satisfaction_score"], errors="coerce")
    df["satisfaction_missing"] = df["satisfaction_score"].isna().astype(int)
    log.append(("keep-nan", "satisfaction_score", "survey non-response; imputation invents an attitude"))

    # 8. zip_area / channel: 1.5% missing, but the missing group visits 26% vs 16%
    #    An explicit unknown level is kept instead of the mode
    for c in ["zip_area", "channel"]:
        df[c] = df[c].fillna("Unknown")
        log.append(("unknown-level", c, "missing group visits 26% vs 16%; blank is a valid level"))

    # 9. delivery / marriage: numeric codes that are categories; averaging 1/2/3 is meaningless
    for c in ["delivery", "marriage"]:
        s = pd.to_numeric(df[c], errors="coerce").astype("Int64").astype(str)
        df[c] = s.replace({"<NA>": "Unknown", "nan": "Unknown"}).astype("category")
        log.append(("measurement-level", c, "numeric codes are categories; averaging 1/2/3 is meaningless"))

    # 10. Binary flags to Int64 (nullable integer, avoids float casting)
    for c in BIN_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # 11. Nominal categories declared explicitly
    for c in CAT_COLS + ["email_segment"]:
        df[c] = df[c].astype("category")

    # 12. The label exists in the training file only
    if is_training:
        df["visit"] = pd.to_numeric(df["visit"]).astype(int)

    return df, ids, log

## 5 · Audit on the raw frames

Text-level overview first — it has to run before the numeric cast, otherwise `'3) 200 - 350'` and
`'1,000 +'` are converted away and the encoding can no longer be inspected.

In [11]:
    check_data_dir()
    raw_train = load(TRAIN_FILE)
    raw_deploy = load(DEPLOY_FILE)

    # Text-level first impression: must run before the numeric cast,
    # otherwise '3) 200 - 350' and '1,000 +' are converted away
    overview(raw_train, "campaign_data")
    overview(raw_deploy, "next_campaign")


=== campaign_data ===
shape: 52,000 rows x 21 columns
  Customer_ID recency purchase_segment purchase mens womens  zip_area new_customer       channel email_segment age dependent account employed phone delivery marriage payment_card satisfaction_score spend visit
0    30000001      12       1) 0 - 100    81.61    1      0     Urban            0         Phone     No E-Mail  42         1       1        1     1        1        0            1                NaN   0.0     0
1    30000004     NaN     3) 200 - 350   258.74    1      0     Urban            0  Multichannel   Mens E-Mail  38         1       1        1     1        3        0            1                  8   0.0     0
2    30000005       6     3) 200 - 350   239.96    1      0  Suburban            0         Phone     No E-Mail  26         0       1        1     1        1        1            1                NaN   0.0     0

--- Column overview ---
                   dtype  n_unique  n_missing  pct_missing                      

,dtype,n_unique,n_missing,pct_missing,top_3_values
column,,,,,
Customer_ID,str,12000,0,0.0,30000002(1) | 30000003(1) | 30000020(1)
recency,str,12,894,7.4,"10(1,349) | 1(1,331) | 9(1,126)"
purchase_segment,str,7,7,0.1,"1) 0 - 100(4,177) | 2) 100 - 200(2,649) | 3) 2..."
purchase,str,9994,0,0.0,29.09(9) | 30.31(9) | 30.78(8)
mens,str,2,0,0.0,"1(6,649) | 0(5,351)"
womens,str,2,0,0.0,"1(6,587) | 0(5,413)"
zip_area,str,3,164,1.4,"Suburban(5,293) | Urban(4,810) | Rural(1,733)"
new_customer,str,2,0,0.0,"1(6,029) | 0(5,971)"
channel,str,3,146,1.2,"Phone(5,281) | Web(5,167) | Multichannel(1,406)"


### Numeric summary, shapes, and the training/deployment gap

In [12]:
    for df in (raw_train, raw_deploy):
        for c in NUM_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    raw_train["visit"] = raw_train["visit"].astype(int)

    numeric_summary(raw_train, NUM_COLS + ["visit"])

    print(f"\ntrain {raw_train.shape}  deploy {raw_deploy.shape}")
    print("columns present in train only:", sorted(set(raw_train) - set(raw_deploy)))


--- Numeric summary ---
                        n    mean      sd   min    p01  median      p99      max  skew  n_far_3iqr
recency             48238    6.04    3.45   1.0   1.00    6.00    12.00    12.00  0.04           0
purchase            52000  241.62  256.75  25.0  27.54  156.82  1229.36  3387.49  2.43         760
age                 49987   35.37    8.94  19.0  22.00   34.00    58.00    62.00  0.59           0
satisfaction_score  20918    7.00    1.45   2.0   4.00    7.00    10.00    10.00 -0.08           0
spend               51965   15.26   42.98   0.0   0.00    0.00   201.49   499.00  3.94        8449
visit               52000    0.16    0.37   0.0   0.00    0.00     1.00     1.00  1.82        8524

train (52000, 21)  deploy (12000, 20)
columns present in train only: ['visit']


### `purchase_segment` label vs the band implied by the amount

In [13]:
    print("\n--- purchase_segment consistency ---")
    for nm, d in (("campaign_data", raw_train), ("next_campaign", raw_deploy)):
        cons = segment_consistency(d)
        print(f"  {nm:14s} checked {cons['checked']:,} | mismatch {cons['mismatch']} "
              f"| label missing {cons['label_missing']}")


--- purchase_segment consistency ---
  campaign_data  checked 51,977 | mismatch 0 | label missing 23
  next_campaign  checked 11,993 | mismatch 0 | label missing 7


### Category levels actually present

In [14]:
    print("\n--- Category levels ---")
    print(raw_train[["zip_area", "channel", "email_segment", "delivery", "marriage"]]
          .apply(lambda s: sorted(s.dropna().unique())).to_string())


--- Category levels ---
   zip_area       channel  email_segment delivery marriage
0     Rural  Multichannel    Mens E-Mail        1        0
1  Suburban         Phone      No E-Mail        2        1
2     Urban           Web  Womens E-Mail        3        2


### Duplicates and ID integrity

In [15]:
    print("\n--- Duplicates and ID integrity ---")
    print("duplicate Customer_ID:", int(raw_train["Customer_ID"].duplicated().sum()))
    print("ID overlap between files:", len(set(raw_train["Customer_ID"]) & set(raw_deploy["Customer_ID"])))


--- Duplicates and ID integrity ---
duplicate Customer_ID: 0
ID overlap between files: 0


### Missingness profile

Reads column by column: the missing group versus the observed group. `gap_pp` is the visit-rate gap,
`flag_auc` asks how well the *missingness flag alone* predicts the label, and `smd_purchase` is the
standardised mean difference (−0.48 / +0.44 / +0.61 / +0.44 / +0.30, all beyond the 0.1 MAR convention).

In [16]:
    missing_profile(raw_train)


--- Missingness profile (missing vs observed) ---
                    n_missing   pct  visit_missing  visit_observed  gap_pp  flag_auc  new_share  new_share_obs  high_band  high_band_obs  smd_purchase  spend_if_visit  spend_if_visit_obs
column                                                                                                                                                                                    
satisfaction_score      31082  59.8           13.7            20.4    -6.6     0.442       48.8           52.3        8.1           19.5         -0.48            89.5                96.6
recency                  3762   7.2           30.6            15.3    15.4     0.538       63.1           49.2       25.7           11.7          0.44            95.2                92.7
age                      2013   3.9           33.0            15.7    17.3     0.523       86.1           48.8       33.8           11.9          0.61            95.6                92.8
zip_area      

,n_missing,pct,visit_missing,visit_observed,gap_pp,flag_auc,new_share,new_share_obs,high_band,high_band_obs,smd_purchase,spend_if_visit,spend_if_visit_obs
column,,,,,,,,,,,,,
satisfaction_score,31082,59.8,13.7,20.4,-6.6,0.442,48.8,52.3,8.1,19.5,-0.48,89.5,96.6
recency,3762,7.2,30.6,15.3,15.4,0.538,63.1,49.2,25.7,11.7,0.44,95.2,92.7
age,2013,3.9,33.0,15.7,17.3,0.523,86.1,48.8,33.8,11.9,0.61,95.6,92.8
zip_area,779,1.5,26.2,16.2,9.9,0.505,78.0,49.8,27.2,12.5,0.44,91.5,93.1
channel,759,1.5,25.8,16.3,9.6,0.505,76.5,49.8,23.6,12.6,0.30,93.9,93.0


### Treatment arms and class imbalance

In [17]:
    print("\n--- Treatment arms and class imbalance ---")
    print(raw_train["email_segment"].value_counts().to_string())
    print(f"positive rate {raw_train['visit'].mean() * 100:.2f}%  "
          f"imbalance 1:{1 / raw_train['visit'].mean() - 1:.1f}  "
          f"break-even lift {BREAKEVEN_PP:.1f}pp")


--- Treatment arms and class imbalance ---
email_segment
No E-Mail        17334
Mens E-Mail      17334
Womens E-Mail    17332
positive rate 16.39%  imbalance 1:5.1  break-even lift 12.5pp


In [18]:
    # Missing rates should be equal across arms; inequality means missingness
    # is related to the email received (confounding between arm and data quality)
    print("\nMissing rate by arm (should be equal):")
    for c in ["zip_area", "channel", "recency", "age", "satisfaction_score"]:
        rates = [round(float(raw_train.loc[raw_train["email_segment"] == a, c].isna().mean() * 100), 2)
                 for a in ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]]
        print(f"  {c:20} {rates}")


Missing rate by arm (should be equal):
  zip_area             [1.52, 1.48, 1.5]
  channel              [1.55, 1.4, 1.43]
  recency              [7.32, 7.36, 7.02]
  age                  [3.8, 3.87, 3.95]
  satisfaction_score   [60.04, 59.74, 59.54]


## 6 · Apply the cleaning and print the decision log

In [19]:
    train_clean, _, log1 = clean(raw_train, True)
    deploy_clean, _, log2 = clean(raw_deploy, False)

    print("\n--- Cleaning log ---")
    for step, col, why in log1:
        print(f"  {step:18} {col:20} {why}")

    print(f"\ncleaned: {train_clean.shape} and {deploy_clean.shape}")
    print("missing kept by design: recency", int(train_clean["recency"].isna().sum()),
          "| age", int(train_clean["age"].isna().sum()),
          "| satisfaction", int(train_clean["satisfaction_score"].isna().sum()))


--- Cleaning log ---
  delete             account              zero variance: every customer has an account
  delete             spend                post-outcome variable: spend>0 implies visit
  delete             purchase_segment     coarse copy of purchase; ordinal rebuilt for reporting only
  keep-nan           recency              missing concentrated in high-value customers; imputation forbidden
  keep-nan           age                  structural missingness: new customers with incomplete profiles
  keep-nan           satisfaction_score   survey non-response; imputation invents an attitude
  unknown-level      zip_area             missing group visits 26% vs 16%; blank is a valid level
  unknown-level      channel              missing group visits 26% vs 16%; blank is a valid level
  measurement-level  delivery             numeric codes are categories; averaging 1/2/3 is meaningless
  measurement-level  marriage             numeric codes are categories; averaging 1/2/3 is mean

## 7 · Feature set handed to Phase 2

`Customer_ID` is held out for submission, `purchase_segment_ord` is kept only for reporting (it is a
coarse copy of `purchase`), and `email_segment` is the treatment, not a covariate.

In [20]:
    FEATURES = ["recency", "recency_missing", "purchase", "mens", "womens", "zip_area",
                "new_customer", "channel", "age", "age_missing", "dependent", "employed",
                "phone", "delivery", "marriage", "payment_card",
                "satisfaction_score", "satisfaction_missing"]
    print(f"\nmodel features ({len(FEATURES)}):", FEATURES)
    print("held out: Customer_ID (submission), purchase_segment_ord (report), email_segment (treatment)")


model features (18): ['recency', 'recency_missing', 'purchase', 'mens', 'womens', 'zip_area', 'new_customer', 'channel', 'age', 'age_missing', 'dependent', 'employed', 'phone', 'delivery', 'marriage', 'payment_card', 'satisfaction_score', 'satisfaction_missing']
held out: Customer_ID (submission), purchase_segment_ord (report), email_segment (treatment)


---

Every number in the Phase 1 report is a line of this output. If a claim cannot be read off these tables,
it does not go into the report.